# 01 — Data and study understanding

**Question this notebook answers:** what is in this dataset, what does each variable mean, and what analysis does it make possible?

The scientific context comes from Koudou et al. (2022), *Malaria Journal* 21:85, which established the accelerometer methodology. This extract is a separate, later tagged dataset — the numbers here are not a re-analysis of that paper.

In [ ]:
import sys, warnings
sys.path.insert(0, '../src')
warnings.filterwarnings('ignore')
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
pd.set_option('display.width', 200)
from smartnet import config
from smartnet.data import loader

df = loader.load_analysis_frame()
print(df.shape)
df.head(3)

## Label schemes

Four nested schemes. Coarser schemes collapse classes of finer ones, so they answer progressively easier questions.

In [ ]:
for col, names in config.LABEL_MAPS.items():
    vc = df[col].value_counts().sort_index()
    print(f'--- {col}')
    for k, v in vc.items():
        print(f'   {names[int(k)]:<16} {v:5d}  ({100*v/len(df):5.1f}%)')

The imbalance is the central design constraint: **Enter (47) and Exit (44) are the rarest classes and the most operationally interesting ones.** Any evaluation reported as overall accuracy will be dominated by Nothing and Put up.

## Feature structure

Every row carries a 21-second context window: the tagged second, the 10 preceding seconds, the 10 following seconds, plus aggregates over those windows.

In [ ]:
for block, cols in config.FEATURE_BLOCKS.items():
    print(f'{block:<26} {len(cols):3d} features')
print()
print('Current epoch :', config.CURRENT_EPOCH_FEATURES)
print('Aggregates    :', config.WINDOW_AGGREGATE_FEATURES)

## Temporal structure

This determines the entire experimental design, so it is worth establishing carefully.

In [ ]:
d = df.sort_values('timestamp')
gaps = d['gap_seconds'].dropna()
print('Period      :', df.timestamp.min(), '->', df.timestamp.max())
print('Recording days:', df.session_date.nunique())
print()
print('Gap between consecutive epochs (seconds):')
print(gaps.value_counts().head(6).to_string())
print(f'\n{(gaps==1).sum()} of {len(gaps)} gaps are exactly 1 second')

In [ ]:
events = loader.event_summary(df, 'motion_5cat')
print(f'{len(events)} contiguous events')
print(events.n_epochs.describe().to_string())
events.head()

**Key finding.** Labelled motions are recorded as contiguous runs of 1-second epochs, up to 16 seconds long. Every run is label-pure — no event spans two classes. This gives a legitimate grouping unit for cross-validation, which notebook 05 relies on.

In [ ]:
loader.assert_events_are_label_pure(df)
print('All events label-pure.')

## What this dataset does *not* contain

There is **no participant, device or bed-net identifier**. Consequences:

1. Grouping can be done by event and by recording day, but **not by person**.
2. Performance on a *new participant* — the deployment-relevant question — cannot be estimated.
3. The adult/child subgroup comparison reported in the published study **cannot be reproduced**.

These are stated as limitations throughout rather than worked around.

In [ ]:
missing_ids = [c for c in ['participant_id','device_id','net_id','subject'] if c in df.columns]
print('Identifier columns present:', missing_ids or 'none')